# Demo 6: Bouw je eigen RL-toepassing

Jij beschrijft het probleem, de robot vindt de policy met Q-learning. Je past alleen het probleem aan, het leren zelf gebeurt achter de schermen.

| Deel | Wat je doet |
|---|---|
| **A. Definiëren** | toestanden, acties, overgangskansen, beloningen |
| **B. Instellen** | leersnelheid, verkenning, discount |
| **C. Trainen** | een beetje trainen en opslaan, dan laden, verder trainen en weer opslaan |
| **D. Veranderen** | verander de beloningen (of iets anders) en laat de opgeslagen robot opnieuw leren |
| **E. Je eigen probleem** | vervang het voorbeeld door je eigen probleem |

**Het voorbeeld** 3 toestanden (A, B, C) en 3 acties (1, 2, 3). Alles staat in een matrix van wat er na elke actie gebeurt, plus een beloning per toestand en een beloning per actie.

**Zo gebruik je het:** klik één keer op *Run → Run All Cells* om het hele verhaal te zien. Pas daarna een cel aan, voer hem uit en voer de trainingscellen uit die je nodig hebt. Trainen gebruikt altijd de definities en instellingen die je het laatst hebt uitgevoerd.

## Zet de robot aan

Voer deze cel eerst uit. Het leren, controleren en tekenen gebeurt achter de schermen, hier hoef je niets aan te passen.

In [1]:
from rl_demos.build_your_own import *

connect(globals())    # lets the robot read your cells

## A. Definieer het probleem

### 1. Toestanden en acties

Een toestand moet alles bevatten wat ertoe doet voor wat er daarna gebeurt. Hier is elke actie in elke toestand mogelijk.

Als het helpt, lees het voorbeeld als een klant: A is *tevreden*, B is *neutraal* en C is *ontevreden*. Actie 1 is *niets doen*, 2 is *kleine moeite* en 3 is *grote moeite*.

In [2]:
STATES = ['A', 'B', 'C']
ACTIONS = [1, 2, 3]

START = {'A': 1.0}     # where an episode starts, with which probability

### 2. Overgangskansen: een matrix van 3 × 3 × 3

We geven hier aan hoe de omgeving veranderd waar de robot in leeft
De robot ziet deze getallen nooit. Ze zijn de gesimuleerde wereld waarin deze oefent.

In [3]:
TRANSITIONS = [
    #   to:   A    B    C
    [                         # from A
        [0.7, 0.2, 0.1],      #   action 1
        [0.9, 0.1, 0.0],      #   action 2
        [1.0, 0.0, 0.0],      #   action 3
    ],
    [                         # from B
        [0.1, 0.5, 0.4],
        [0.5, 0.4, 0.1],
        [0.8, 0.2, 0.0],
    ],
    [                         # from C
        [0.0, 0.1, 0.9],
        [0.1, 0.4, 0.5],
        [0.5, 0.4, 0.1],
    ],
]

show_diagram()      # draws these probabilities: one picture per action

### 3. Beloningen: een beloning per toestand en een beloning per actie

De robot wil zo veel mogelijk beloning verzamelen. Elke stap levert twee dingen op:
- een vaste beloning voor elke stap die je in een toestand bent
- de beloning van de actie die je kiest (moeite kost iets, dus die is negatief)


In [4]:
STATE_REWARDS = [10, 8, 0]      # every step in A, B, C
ACTION_REWARDS = [0, -4, -9]    # action 1, 2, 3

REWARDS = combine_rewards(ACTION_REWARDS, STATE_REWARDS)

show_diagram()      # the same picture, now with the rewards on the arrows

## B. Hoe de robot leert

| Instelling | Laag | Hoog |
|---|---|---|
| **Leersnelheid** | langzaam maar stabiel | snel maar onrustig |
| **Verkenning** | vindt misschien nooit iets beters, merkt veranderingen traag op | blijft dingen proberen waarvan bekend is dat ze slecht zijn |
| **Discount** | alleen de volgende beloning telt | de lange termijn telt |

In [5]:
LEARNING_RATE = 0.02   # above 0, up to 1
EXPLORATION = 0.2      # 0 to 1: how often it tries a random action
DISCOUNT = 0.9         # 0 up to, not including, 1
MAX_STEPS = 50         # steps per practice episode
SEED = 0               # change it to see whether your result was luck

### Controleer je probleem

Voer dit uit na elke wijziging in deel A. Het noemt fouten, zoals een regel die niet optelt tot 1 of een matrix van de verkeerde grootte, en toont beide matrices als één tabel.

In [6]:
show_problem()

state,action,reward,to A,to B,to C
A,1,10,70%,20%,10%
A,2,6,90%,10%,0%
A,3,1,100%,0%,0%
B,1,8,10%,50%,40%
B,2,4,50%,40%,10%
B,3,-1,80%,20%,0%
C,1,0,0%,10%,90%
C,2,-4,10%,40%,50%
C,3,-9,50%,40%,10%


## C. Trainen, opslaan, verder trainen

### 4. Begin met een lege robot

Voer deze cel opnieuw uit als je wilt dat hij alles vergeet.

In [7]:
robot = new_robot()

### 5. Train tot een bepaald punt en sla op

Maar een beetje oefening, dus verwacht fouten. Het bestand bevat de policy en de Q-matrix: de Q-matrix heb je nodig om verder te trainen.

In [8]:
train(robot, episodes=20, save_as='v1')
show_policy(robot)

Trained 20 episodes (0 -> 20 in total)
Saved saved_policies/v1.json (replaced the older one)


state,1,2,3,robot does,perfect policy
A,19.955 tries,33.0688 tries,15.854 tries,2,1
B,2.58 tries,20.6134 tries,4.111 tries,2,2
C,0.027 tries,0.26 tries,1.717 tries,3,3


### 6. Laad hem en train verder

Wil je nog verder, verander dan de namen: laad `'v2'` en sla op als `'v2b'`.

In [9]:
robot = load('v1')
train(robot, episodes=5000, save_as='v2')
show_policy(robot)
show_learning({'robot': robot})

Loaded v1 (20 episodes of experience)
Trained 5,000 episodes (20 -> 5,020 in total)
Saved saved_policies/v2.json (replaced the older one)


state,1,2,3,robot does,perfect policy
A,"68.2136,559 tries","66.017,460 tries","61.911,004 tries",1,1
B,"58.44,746 tries","59.751,202 tries","58.45,063 tries",2,2
C,"43.51,751 tries","45.21,969 tries","47.521,246 tries",3,3


robot,at start,highest,at end
robot,46,69,69


## D. De wereld verandert: opnieuw leren

### 7. Verander iets

In het voorbeeld levert een stap in C ineens 8 op in plaats van 0, net zoveel als B. Is moeite dan nog de moeite waard? Je kunt ook elke cel in deel A of B aanpassen en uitvoeren, bijvoorbeeld de kansen of `EXPLORATION`.

Deze cel verandert `REWARDS`. Om dat terug te draaien voer je de beloningscel (3) opnieuw uit.

In [12]:
STATE_REWARDS = [10, 8, 8]      # C: 0 -> 8
REWARDS = combine_rewards(ACTION_REWARDS, STATE_REWARDS)
print('Rewards per state (rows A, B, C) and action (columns 1, 2, 3):', REWARDS)

Rewards per state (rows A, B, C) and action (columns 1, 2, 3): [[10, 6, 1], [8, 4, -1], [8, 4, -1]]


### 8. Hoe goed is de opgeslagen policy nu?

Hij heeft nog niets over de verandering geleerd.

In [13]:
compare({'v2 (saved before the change)': load('v2')})

Loaded v2 (5,020 episodes of experience)


policy,A,B,C,score
v2 (saved before the change),1,2,3,76
perfect,1,1,1,86
random,any,any,any,52


### 9. Leer opnieuw vanaf de opgeslagen policy

`v2` blijft zoals hij was, de opnieuw getrainde robot wordt opgeslagen als `v3`. In de grafiek zie je de daling op het moment van de verandering, en het herstel.

In [14]:
robot = load('v2')
train(robot, episodes=5000, save_as='v3')
show_policy(robot)
show_learning({'robot': robot})

Loaded v2 (5,020 episodes of experience)
Changed since the last training: REWARDS -> relearning from what it already knew
Trained 5,000 episodes (5,020 -> 10,020 in total)
Saved saved_policies/v3.json (replaced the older one)


state,1,2,3,robot does,perfect policy
A,"86.2202,516 tries","83.022,545 tries","78.616,071 tries",1,1
B,"81.849,731 tries","79.057,993 tries","75.28,648 tries",1,1
C,"80.8102,869 tries","77.69,886 tries","74.530,741 tries",1,1


robot,at start,highest,at end
robot,46,86,86
